# A1 — OLS Cross-Section

**Famiglia A** | Una riga per contatore | Baseline della tesi

```
pct_silente_i = β₀ + β·X_i + ε_i
```
- Variabile dipendente: proporzione di finestre silenti [0, 1]
- Include variabili statiche + aggregate dinamiche
- **Prerequisito**: eseguire `00_preparazione_dati/01_Preparazione_dati_v2.ipynb` per generare i parquet

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 1 — Setup: mount Drive + clone repo + caricamento parquet
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")

import subprocess, os, sys

REPO_URL  = "https://github.com/swaggincoffee-bit/Tesi.git"
REPO_PATH = "/content/Tesi"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
sys.path.insert(0, REPO_PATH)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

from src.config import (
    COL_DATA_INST, COL_ANNO_COSTR, COL_CONSUMO,
    COL_MODELLO, COL_TECN_COM, COL_COSTRUTTORE,
    FNAME_CS,
)

PROC = "/content/drive/MyDrive/Contatori/OUTPUT/"
OUT  = "/content/drive/MyDrive/Contatori/OUTPUT/"

df_cs = pd.read_parquet(PROC + FNAME_CS)

print(f"df_cs caricato: {df_cs.shape}")
display(df_cs.head(3))

In [2]:
# CELLA 2 - Feature engineering

from sklearn.preprocessing import LabelEncoder

df_lett_re = pd.read_parquet(PROC + "df_lett_re.parquet")
ref_date = pd.to_datetime(df_lett_re["data_lettura"]).max()
del df_lett_re

# Categoriche -> encoded
for short, col in {"costruttore_enc":  "costruttore",
                   "modello_enc":      "modello",
                   "tecnologia_enc":   "tecnologia",
                   "comune_enc":       "comune",
                   "accessibile_enc":  "accessibile",
                   "telegestione_enc": "telegestione"}.items():
    df_cs[short] = LabelEncoder().fit_transform(df_cs[col].fillna("N/A"))

FEATURES = [
    "anni_da_costruzione",
    "consumo_annuo",
    "gg_da_installazione",
    "firmware_num",
    "gg_da_ult_com",
    "gg_da_ult_mis",
    "costruttore_enc",
    "modello_enc",
    "tecnologia_enc",
    "comune_enc",
    "accessibile_enc",
    "telegestione_enc",
    "pct_err",
    *[f"diag_bit_{i:02d}" for i in range(16)],
]
TARGET = "pct_silente"

df_model = df_cs[FEATURES + [TARGET]].dropna()
print(f"Osservazioni: {len(df_model):,}  (NaN scartati: {len(df_cs)-len(df_model):,})")
display(df_model[FEATURES].describe().round(2))

In [3]:
print(df_cs.columns.tolist())

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — OLS con statsmodels
# ══════════════════════════════════════════════════════════════════════════════

X = sm.add_constant(df_model[FEATURES])
y = df_model[TARGET]

model = sm.OLS(y, X).fit(cov_type="HC3")  # errori robusti eteroschedasticità
print(model.summary())


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — OLS con statsmodels
# ══════════════════════════════════════════════════════════════════════════════

X = sm.add_constant(df_model[FEATURES])
y = df_model[TARGET]

model = sm.OLS(y, X).fit(cov_type="HC3")  # errori robusti eteroschedasticità
print(model.summary())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 5 — Salvataggio risultati
# ══════════════════════════════════════════════════════════════════════════════

risultati = model.summary2().tables[1].reset_index()
risultati.columns = ["variabile", "coef", "std_err", "t", "p_value", "ci_low", "ci_high"]
risultati.to_csv(OUT + "A1_coefficienti_OLS.csv", index=False)

print("✅ Salvati:")
print("  A1_diagnostica_residui.png")
print("  A1_coefficienti_OLS.csv")
display(risultati)

In [ ]:
df_lett_re[COL_STATO_LETT].value_counts()
df_lett_re[COL_DIAGNOSTICA].value_counts()